In [1]:
%pip install ollama


   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------------------- ------- 1.6/2.0 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 7.2 MB/s eta 0:00:00
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.19
    Uninstalling pydantic-1.10.19:
      Successfully uninstalled pydantic-1.10.19


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.95.2 requires pydantic!=1.7,!=1.7.1,!=1.7.2,!=1.7.3,!=1.8,!=1.8.1,<2.0.0,>=1.6.2, but you have pydantic 2.11.1 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import ollama

def query_deepseek(prompt):
    response = ollama.chat(
        model="deepseek-r1:1.5b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

# Test it
resume_text = query_deepseek("Generate a well-structured ATS-friendly resume for a Software Engineer.")
print(resume_text)


<think>
Okay, so I need to help someone generate a well-structured ATS-friendly resume for a Software Engineer. Let me think about how to approach this.

First, I should understand the purpose of an ATS-resizable resume. The user provided an example that's pretty detailed, covering sections like Objective, Experience, Education, Skills, Certifications, and References. That seems comprehensive, but maybe I can expand on each section or make it more tailored for a Software Engineer role.

Let me break down what makes a resume ATS-friendly. They should be easy to update as information changes, well-organized with clear sections, and include necessary details like education, work experience, certifications, skills, and references. I'll need to structure this in a way that's both professional and user-friendly.

Starting with the Objective: It needs to summarize what I'm looking for in a job. For a Software Engineer, it should highlight passion for programming, technical skills relevant to 

In [22]:
import ollama
import json
import re

# User-provided resume data
user_data = {
    "name": "John Doe",
    "email": "johndoe@example.com",
    "phone": "123-456-7890",
    "experience": [
        {
            "role": "Software Engineer",
            "company": "Google",
            "startDate": "01-01-2020",
            "endDate": "31-12-2023"
        }
    ],
    "education": [
        {
            "degree": "B.Tech in CS",
            "institution": "MIT",
            "year": "2022"
        }
    ],
    "skills": ["Python", "Machine Learning", "Cloud Computing", "Leadership", "Communication"]
}

# Strict prompt enforcing provided details
prompt = f"""
You are an AI that generates **STRICTLY VALID JSON ONLY** for a resume.
**DO NOT MODIFY OR OMIT ANY PROVIDED DATA.**  
NO MARKDOWN. NO EXPLANATIONS. **ONLY JSON OUTPUT!**

---
🚨 **STRICT RULES** 🚨
1️⃣ **Use exact personal details as provided. DO NOT CHANGE NAME, EMAIL, PHONE, OR COMPANY.**
2️⃣ **Experience must exactly match the provided role, company, and dates.**
3️⃣ **Summary must be 4-5 sentences based on given skills and experience.**
4️⃣ **Each experience must have at least 3 bullet points describing work done.**
5️⃣ **Education must include a one-sentence description.**
6️⃣ **All skills must be categorized under:**
   - `"Industrial Knowledge"`  
   - `"Tools & Technologies"`  
   - `"Soft Skills"`

---
✅ **EXPECTED JSON FORMAT** (STRICTLY FOLLOW THIS)
{{
    "name": "{user_data['name']}",
    "email": "{user_data['email']}",
    "phone": "{user_data['phone']}",
    "summary": "FILL_THIS_IN (4-5 sentences about experience, skills, and achievements).",
    "experience": [
        {{
            "role": "{user_data['experience'][0]['role']}",
            "company": "{user_data['experience'][0]['company']}",
            "startDate": "{user_data['experience'][0]['startDate']}",
            "endDate": "{user_data['experience'][0]['endDate']}",
            "description": [
                "FILL_THIS_IN (Key achievement or responsibility).",
                "FILL_THIS_IN (Another key contribution).",
                "FILL_THIS_IN (One more impactful point)."
            ]
        }}
    ],
    "education": [
        {{
            "degree": "{user_data['education'][0]['degree']}",
            "institution": "{user_data['education'][0]['institution']}",
            "year": "{user_data['education'][0]['year']}",
            "description": "FILL_THIS_IN (One sentence about coursework or achievements)."
        }}
    ],
    "skills": {{
        "Industrial Knowledge": ["FILL_THIS_IN"],
        "Tools & Technologies": {json.dumps([s for s in user_data["skills"] if s.lower() not in ["leadership", "communication"]])},
        "Soft Skills": {json.dumps([s for s in user_data["skills"] if s.lower() in ["leadership", "communication"]])}
    }}
}}

---
🚨 **IMPORTANT:** 🚨
- **DO NOT RETURN ANYTHING ELSE EXCEPT PURE JSON.**
- **NO MARKDOWN (` ```json `), NO TEXT, NO EXPLANATIONS, NO COMMENTS, JUST JSON!**
"""

# Query DeepSeek-R1
response = ollama.chat(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": prompt}])

# Extract text safely
output_text = response.get("message", {}).get("content", "").strip()

if not output_text:
    print("❌ Error: No 'message' or 'content' found in response.")
else:
    # Remove unwanted AI-generated sections
    output_text = re.sub(r"<think>.*?</think>", "", output_text, flags=re.DOTALL).strip()

    # Validate JSON output
    try:
        resume_json = json.loads(output_text)
        print("✅ Successfully Parsed JSON:", json.dumps(resume_json, indent=4))
    except json.JSONDecodeError:
        print("❌ Error: DeepSeek returned invalid JSON.")
        print("Raw Output:", output_text)


❌ Error: DeepSeek returned invalid JSON.
Raw Output: {
    "name": "John Doe",
    "email": "johndoe@example.com",
    "phone": "123-456-7890",
    "summary": "FILL_THIS_IN (4-5 sentences about experience, skills, and achievements).",
    "experience": [
        {
            "role": "Software Engineer",
            "company": "Google",
            "startDate": "01-01-2020",
            "endDate": "31-12-2023",
            "description": [
                "Developed recommendation system using machine learning.", 
                "Wrote Python and Java EE code for data processing.", 
                "Collaborated on cloud service integration with AWS."
            ]
        },
        {
            "role": "Software Engineer",
            "company": "Microsoft", 
            "startDate": "01-01-2023", 
            "endDate": "30-12-2024", 
            "description": [
                "Developed cloud-native applications using Azure," 
                "Designed user interfaces with mode